### 1. Setup:
Importing three tools: Roboflow, Ultralytics, and Gradio. Plus a GPU check. 
Our team choose Ultralytics "8.4.120" version.

In [ ]:
ULTRALYTICS_VERSION = "8.4.120"

try:
    import google.colab  # noqa: F401
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    import subprocess
    # roboflow is left unpinned on purpose. It talks to a hosted service, so the
    # version that matches the service is whichever one is current.
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "roboflow"],
        check=True,
    )

print("Running on Colab:", RUNNING_ON_COLAB)

In [ ]:
import json
import os
import random

# Imports Gradio, used to create a web interface for the model for deployment.
import gradio as gr
from PIL import Image, UnidentifiedImageError

# Imports PyTorch (torch), the underlying deep learning framework
import torch
from ultralytics import YOLO

print("torch    :", torch.__version__)
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

if not torch.cuda.is_available():
    print("")
    print("No GPU. In Colab: Runtime, Change runtime type, T4 GPU.")
    print("Training on CPU works but takes far longer than the lab allows.")

### 2. Storing work results in Google Drive:
- PROJECT_ROOT: "/content/drive/MyDrive/C389_CA1".
- WORK_DIR:
 "/content/drive/MyDrive/C389_CA1/runs".

In [ ]:
if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389_CA1"
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

WORK_DIR = os.path.join(PROJECT_ROOT, "runs")
os.makedirs(WORK_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK_DIR    :", WORK_DIR)

### 3. Prepare the dataset from Roboflow:
Enter the API key manually because the notebook is pushed to the team's repository (Public), anyone holding it can read and change our team's project.

To avoid typing the key every time, save it once in Colab's **Secrets** panel, the
key icon on the left, under the name `ROBOFLOW_API_KEY`, and allow this notebook to
read it. The cell uses the secret when it finds one and asks otherwise.


In [ ]:
import re
from getpass import getpass

# Paste the download code from Roboflow between the triple quotes, then delete the
# key so the line reads api_key="". Leave it empty to use the sample set.
ROBOFLOW_CODE = """
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="")
project = rf.workspace("fangs-workspace-hj04l").project("c389-team2-ca1-combined-dataset")
version = project.version(1)
dataset = version.download("yolo26")
"""

YOLO_FORMATS = ("yolov8", "yolo26", "yolov11", "yolov12", "yolov9", "yolov5")

def read_roboflow_code(code):
    """Read the workspace, project, version and format out of Roboflow's code.

    Returns None when nothing was pasted. Raises ValueError when the code still
    holds the private key, or lacks a value the download needs.
    """
    if not code.strip():
        return None
    key = re.search(r"api_key\s*=\s*[\"']([^\"']*)[\"']", code)
    if key and key.group(1).strip():
        raise ValueError(
            "The pasted code still holds your private API key. Delete it so the "
            "line reads api_key=\"\", then run the cell again. The key is asked "
            "for when the cell runs, so it is never saved in the notebook.")
    found = {}
    for name, pattern in (("workspace", r"\.workspace\(\s*[\"']([^\"']+)[\"']"),
                          ("project", r"\.project\(\s*[\"']([^\"']+)[\"']"),
                          ("version", r"\.version\(\s*(\d+)\s*\)"),
                          ("format", r"\.download\(\s*[\"']([^\"']+)[\"']")):
        match = re.search(pattern, code)
        if not match:
            raise ValueError(
                "No %s in the pasted code. Copy the whole block Roboflow shows "
                "under Show download code." % name)
        found[name] = match.group(1)
    found["version"] = int(found["version"])
    # YOLOv8, YOLO26 and the other YOLO text formats write the same files: one
    # label file per image and a data.yaml. Anything else is a different layout.
    if found["format"] not in YOLO_FORMATS:
        raise ValueError(
            "The code downloads the %r format. Choose YOLO26 in the Download "
            "Dataset window and copy the code again." % found["format"])
    return found

def roboflow_key():
    """The private key from Colab Secrets, or else from a hidden prompt."""
    if RUNNING_ON_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("ROBOFLOW_API_KEY").strip()
        except Exception:
            pass  # no secret saved, or this notebook is not allowed to read it
    return getpass("Roboflow private API key: ").strip()

ROBOFLOW = read_roboflow_code(ROBOFLOW_CODE)

if ROBOFLOW:
    from roboflow import Roboflow

    print("workspace %(workspace)s, project %(project)s, version %(version)d" % ROBOFLOW)
    project = Roboflow(api_key=roboflow_key()).workspace(
        ROBOFLOW["workspace"]).project(ROBOFLOW["project"])
    export = project.version(ROBOFLOW["version"]).download(
        ROBOFLOW["format"], location=os.path.join(WORK_DIR, "dataset"), overwrite=True
    )
    DATASET_DIR = export.location
    USING_SAMPLE_DATA = False
else:
    if not os.path.isdir(SAMPLE_DIR):
        raise FileNotFoundError(
            "Nothing was pasted and there is no sample set at %s.\n"
            "Paste the Roboflow download code, or run scripts/make_sample_data.py."
            % SAMPLE_DIR
        )
    DATASET_DIR = SAMPLE_DIR
    USING_SAMPLE_DATA = True

DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")
print("dataset:", DATASET_DIR)

if USING_SAMPLE_DATA:
    print("")
    print("WARNING: using the drawn sample set. It checks that the pipeline runs.")
    print("Every number below is meaningless about the real problem.")

### 4. The class list:

Read it and look at the order before anything else.

In [ ]:
def read_class_names(yaml_path):
    """Read the ordered class list out of a data.yaml."""
    with open(yaml_path, encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("names:"):
                body = line.split(":", 1)[1].strip().strip("[]")
                return [name.strip().strip("'\"") for name in body.split(",")]
    raise ValueError("No names: line in %s" % yaml_path)


BASELINE_CLASSES = ["person_left", "person_right"]
DATASET_CLASSES = read_class_names(DATA_YAML)

for class_id, name in enumerate(DATASET_CLASSES):
    print("%d  %s" % (class_id, name))

### 5. Making the dataset path unambiguous:

A Roboflow export writes its split paths as `../train/images`. Those are
resolved against the folder the file sits in, so the `..` climbs one level too
high and training stops with a dataset-not-found error naming a path nobody
typed.

The cell below rewrites the file with an absolute `path:` and plain split names.
Nothing about the data changes. It is here because this is the single most common
way a first Colab run fails, and the error message does not say what to do.

In [ ]:
def normalise_data_yaml(dataset_dir):
    """Rewrite data.yaml with an absolute path, keeping the class list.

    Returns the path to the file that was written.
    """
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    names_line = None
    with open(yaml_path, encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("names:"):
                names_line = line.rstrip()

    if names_line is None:
        raise ValueError("No names: line in %s" % yaml_path)

    splits = []
    for key, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
        if os.path.isdir(os.path.join(dataset_dir, folder, "images")):
            splits.append("%s: %s/images" % (key, folder))

    if not any(line.startswith("train:") for line in splits):
        raise FileNotFoundError(
            "No train/images folder under %s. The export did not unpack the way "
            "this notebook expects." % dataset_dir)

    class_count = len(read_class_names(yaml_path))
    with open(yaml_path, "w", encoding="utf-8") as handle:
        print("# Paths rewritten by this notebook so that they resolve from",
              "anywhere.", file=handle)
        print("path:", dataset_dir.replace(os.sep, "/"), file=handle)
        for line in splits:
            print(line, file=handle)
        print("", file=handle)
        print("nc:", class_count, file=handle)
        print(names_line, file=handle)
    return yaml_path


normalise_data_yaml(DATASET_DIR)
print(open(DATA_YAML, encoding="utf-8").read())

### 6. The split Roboflow made:

Roboflow put each image in one of three folders when the version was generated:
`train` is what the model learns from, `valid` is what it is checked against while
it trains, and `test` is kept back for the final score in Section 6.

The cell below counts what landed where. A version generated with no test images
still works: the notebook then reports its final score on `valid` and says so.

In [ ]:
SPLIT_FOLDERS = ["train", "valid", "test"]


def count_images(dataset_dir, split):
    """How many images one split folder holds, 0 when the folder is missing."""
    image_dir = os.path.join(dataset_dir, split, "images")
    if not os.path.isdir(image_dir):
        return 0
    return sum(1 for name in os.listdir(image_dir)
               if name.lower().endswith((".jpg", ".jpeg", ".png")))


SPLIT_COUNTS = {split: count_images(DATASET_DIR, split) for split in SPLIT_FOLDERS}
for split in SPLIT_FOLDERS:
    print("%-6s %4d images" % (split, SPLIT_COUNTS[split]))

if not SPLIT_COUNTS["train"]:
    raise ValueError("The version has no training images. Generate it again in "
                     "Roboflow with images in Train.")

# The split every result below is judged on. Ultralytics calls the valid folder "val".
EVAL_FOLDER = "test" if SPLIT_COUNTS["test"] else "valid"
EVAL_SPLIT = "test" if EVAL_FOLDER == "test" else "val"
if EVAL_FOLDER == "valid":
    print("")
    print("No test images, so the final score is read on valid instead.")

### 7. What a pretrained model cannot tell you

Before training anything, run the stock model. It was trained on COCO, which has
a `person` class and nothing direction aware.

Two images are run: a photograph that ships with Ultralytics, and one image from
this dataset. Watch the class of every detection on the photograph. The boxes
will be in the right places and every one of them will say `person`.

In [ ]:
def image_paths(dataset_dir, split, limit=None):
    """List the images in one split, in filename order."""
    image_dir = os.path.join(dataset_dir, split, "images")
    names = sorted(name for name in os.listdir(image_dir)
                   if name.lower().endswith((".jpg", ".jpeg", ".png")))
    if limit:
        names = names[:limit]
    return [os.path.join(image_dir, name) for name in names]


def to_detections(result, class_names):
    """Turn one Ultralytics result into plain dictionaries.

    The same shape is used in the service in Lesson 4, so the counting code
    written below is the code that ships.
    """
    detections = []
    boxes = result.boxes
    if boxes is None:
        return detections
    for index in range(len(boxes)):
        class_id = int(boxes.cls[index].item())
        name = class_names[class_id] if class_id < len(class_names) else str(class_id)
        detections.append({
            "class_name": name,
            "confidence": round(float(boxes.conf[index].item()), 3),
            "box": [round(float(value), 1) for value in boxes.xyxy[index].tolist()],
        })
    return detections


from ultralytics.utils import ASSETS

BASELINE_MODEL = "yolo26n.pt"
pretrained = YOLO(BASELINE_MODEL)
pretrained_names = [pretrained.names[key] for key in sorted(pretrained.names)]

# A real photograph first, then one image from this dataset. The photograph is
# what carries the point, and it is here because a model trained on photographs
# has never seen a drawing and reports nothing on the sample set.
baseline_paths = [os.path.join(str(ASSETS), "bus.jpg")]
baseline_paths += image_paths(DATASET_DIR, EVAL_FOLDER, limit=1)

for path in baseline_paths:
    result = pretrained.predict(path, conf=0.35, verbose=False)[0]
    detections = to_detections(result, pretrained_names)
    print(os.path.basename(path))
    for detection in detections:
        print("   %-12s %.2f" % (detection["class_name"], detection["confidence"]))
    if not detections:
        print("   nothing detected")

### 8. Training configuration:

Four values to choose and one that is not a choice.

Each of the four costs something. A larger variant is more accurate and slower to
train and to serve. More epochs fit the training set better and eventually fit
its noise. A larger image size helps with people far from the camera, which is
where direction is hardest to read, and uses more memory. Batch size is the first
thing to reduce when the run dies out of memory.

### Epochs is the wrong thing to count:

What the model actually gets is **optimiser steps**, which is the number of
training images divided by the batch size, multiplied by the epochs.

Twenty four images at batch 16 is one and a half steps per epoch, so thirty
epochs is about forty five steps. Forty five steps is enough for the model to
learn roughly where people are, so mAP comes out looking reasonable, and nowhere
near enough for it to become confident about anything. Every prediction then
arrives below any sensible threshold and every count comes out zero.

A small dataset needs more epochs, not fewer. The cell after training prints the
number of steps this configuration will produce, so the figure is visible rather
than inferred.

In [ ]:
MODEL_VARIANT = "yolo26n.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16

In [ ]:
# fliplr is pre-written and must stay at 0.0.
#
# Horizontal flip augmentation mirrors the image and keeps the original label. A
# flipped person_left is a person moving right, still labelled person_left, so
# the model is taught the opposite of the truth on half the data. Nothing in the
# output reveals it: the curves look normal and the score looks reasonable. This
# is the one setting in this notebook that is not a choice.
FLIP_LEFT_RIGHT = 0.0

RUN_NAME = "people_counting_baseline&Twist"

missing = [name for name, value in (
    ("MODEL_VARIANT", MODEL_VARIANT), ("EPOCHS", EPOCHS),
    ("IMAGE_SIZE", IMAGE_SIZE), ("BATCH_SIZE", BATCH_SIZE)) if value is None]
if missing:
    raise ValueError("Set these before launching: %s" % ", ".join(missing))

training_images = len(image_paths(DATASET_DIR, "train"))
steps_per_epoch = max(1, training_images // BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS

print("model variant :", MODEL_VARIANT)
print("epochs        :", EPOCHS)
print("image size    :", IMAGE_SIZE)
print("batch size    :", BATCH_SIZE)
print("fliplr        :", FLIP_LEFT_RIGHT, "(must be 0.0)")
print("")
print("training images  : %d" % training_images)
print("optimiser steps  : %d per epoch, %d in total" % (steps_per_epoch, total_steps))

if total_steps < 300:
    print("")
    print("That is a small number of steps. Expect a model that finds roughly the")
    print("right places and is not confident about any of them, which shows up as")
    print("counts of zero in Section 8. Raise the epochs rather than the threshold.")

### 9. Training:



In [ ]:
model = YOLO(MODEL_VARIANT)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    fliplr=FLIP_LEFT_RIGHT,
    project=WORK_DIR,
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
)

RUN_DIR = os.path.join(WORK_DIR, RUN_NAME)
BEST_WEIGHTS = os.path.join(RUN_DIR, "weights", "best.pt")
print("")
print("weights:", BEST_WEIGHTS)

### 10. Safety Verification & Class Instance Counts
Before evaluating the model, we verify that any new twist classes (`person_with_luggage_left`, `person_with_luggage_right`) were appended to the end of the class list. This guarantees that `person_left` (ID 0) and `person_right` (ID 1) retain their original IDs, preventing label corruption. We also count the bounding box instances across `train`, `valid`, and `test` splits to quantify data availability per class.

In [ ]:
# --- 1. Class List Safety Verification & Definition ---
def extend_class_list(existing_classes, new_classes):
    """Add classes to a class list without disturbing existing IDs."""
    extended = list(existing_classes)
    for name in new_classes:
        if name not in extended:
            extended.append(name)
    return extended

NEW_CLASSES = [name for name in DATASET_CLASSES if name not in BASELINE_CLASSES]
CLASS_NAMES = extend_class_list(BASELINE_CLASSES, NEW_CLASSES)

print("Baseline classes :", BASELINE_CLASSES)
print("New twist classes:", NEW_CLASSES)
print("Extended classes :", CLASS_NAMES)

if CLASS_NAMES[:len(BASELINE_CLASSES)] != BASELINE_CLASSES:
    raise ValueError("Original class positions shifted! Append new classes to the end.")

# --- 2. Instance Counting per Split ---
def instances_per_class(dataset_dir, split, class_names):
    """Count annotated bounding box instances for each class in a split."""
    counts = {name: 0 for name in class_names}
    label_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.isdir(label_dir):
        return counts

    for filename in sorted(os.listdir(label_dir)):
        if not filename.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, filename), encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                class_id = int(line.split()[0])
                if class_id < len(class_names):
                    counts[class_names[class_id]] += 1
    return counts

SPLITS = ["train", "valid", "test"]
INSTANCE_COUNTS = {split: instances_per_class(DATASET_DIR, split, CLASS_NAMES) for split in SPLITS}

print("\n%-28s %7s %7s %7s %7s" % ("Class", "Train", "Valid", "Test", "Total"))
print("-" * 60)
for name in CLASS_NAMES:
    row = [INSTANCE_COUNTS[split][name] for split in SPLITS]
    print("%-28s %7d %7d %7d %7d" % (name, row[0], row[1], row[2], sum(row)))

# --- 3. Model Validation & Per-Class Metrics ---
trained_model = YOLO(BEST_WEIGHTS)
metrics = trained_model.val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False)

PER_CLASS_METRICS = {}
for index, class_id in enumerate(metrics.ap_class_index):
    class_name = CLASS_NAMES[int(class_id)]
    PER_CLASS_METRICS[class_name] = {
        "precision": float(metrics.box.p[index]),
        "recall": float(metrics.box.r[index]),
        "ap50": float(metrics.box.ap50[index]),
    }

print("\n%-28s %9s %7s %8s %9s" % ("Class", "Precision", "Recall", "mAP@.5", "Instances"))
print("-" * 68)
for name in CLASS_NAMES:
    if name in PER_CLASS_METRICS:
        row = PER_CLASS_METRICS[name]
        total_inst = sum(INSTANCE_COUNTS[split][name] for split in SPLITS)
        print("%-28s %9.3f %7.3f %8.3f %9d" %
              (name, row["precision"], row["recall"], row["ap50"], total_inst))

# Save metrics payload to JSON file
metrics_payload = {
    "overall": {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
    },
    "per_class": PER_CLASS_METRICS
}
metrics_file = os.path.join(WORK_DIR, "ca1_metrics.json")
with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print("\nMetrics saved to:", metrics_file)

### 11. Confusion Matrix Visualization
The confusion matrix illustrates class-to-class predictions versus ground truth labels. We display it to analyze misclassifications between direction classes (`person_left` vs. `person_right`), luggage carrier states (`person_with_luggage_left` vs. `person_with_luggage_right`), and background errors.

In [ ]:
import matplotlib.pyplot as plt

confusion_matrix_path = os.path.join(RUN_DIR, "confusion_matrix.png")
if os.path.exists(confusion_matrix_path):
    img = Image.open(confusion_matrix_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("CA1 Luggage Detection Confusion Matrix", fontsize=14)
    plt.show()
else:
    print("Confusion matrix image not found at:", confusion_matrix_path)

### 12. Multi-Class Counting & Threshold Sensitivity
We define the core prediction function `predict_counts` to count objects across all 4 classes at a given confidence threshold. We then evaluate a test image across multiple confidence thresholds (`0.10`, `0.25`, `0.40`, `0.60`, `0.80`) to demonstrate the trade-off between false positive noise and under-counting.

In [ ]:
def predict_counts(image_path, model, conf_threshold=0.40):
    """Run inference on an image and return dictionary counts for all classes."""
    results = model.predict(source=image_path, conf=conf_threshold, verbose=False)
    counts = {name: 0 for name in CLASS_NAMES}
    
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            if cls_id < len(CLASS_NAMES):
                counts[CLASS_NAMES[cls_id]] += 1
    return counts, results[0].plot()

# Find a sample test image
test_image_dir = os.path.join(DATASET_DIR, EVAL_FOLDER, "images")
sample_images = [os.path.join(test_image_dir, f) for f in os.listdir(test_image_dir)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

if sample_images:
    sample_img_path = sample_images[0]
    print("Evaluating threshold sensitivity on sample image:", os.path.basename(sample_img_path))
    print("\n%-12s %s" % ("Threshold", "Extracted Counts"))
    print("-" * 65)
    for conf in [0.10, 0.25, 0.40, 0.60, 0.80]:
        c_dict, _ = predict_counts(sample_img_path, trained_model, conf_threshold=conf)
        print("%-12.2f %s" % (conf, c_dict))

### 13. Deployment Web Service & Gradio Interface
We package the trained model into a web application using Gradio. The service implements input validation (`validate_image`) to handle invalid or missing files with HTTP-style status codes (`200`, `400`, `415`), and returns an annotated image alongside the JSON count reply contract.

In [ ]:
MAX_IMAGE_SIZE_MB = 10.0

def validate_image(image_input):
    """Validate uploaded image input and return appropriate HTTP status code and message."""
    if image_input is None:
        return 400, "Bad Request: No image file provided."
    
    if isinstance(image_input, str):
        if not os.path.exists(image_input):
            return 400, "Bad Request: File path does not exist."
        file_size_mb = os.path.getsize(image_input) / (1024 * 1024)
        if file_size_mb > MAX_IMAGE_SIZE_MB:
            return 413, f"Payload Too Large: File size ({file_size_mb:.1f}MB) exceeds {MAX_IMAGE_SIZE_MB}MB limit."
    return 200, "OK"

def build_reply(image_input, conf_threshold=0.40):
    """Process incoming Gradio request, run inference, and return reply payload."""
    status_code, status_msg = validate_image(image_input)
    
    if status_code != 200:
        empty_counts = {name: 0 for name in CLASS_NAMES}
        payload = {
            "status": status_code,
            "message": status_msg,
            "counts": empty_counts
        }
        return None, json.dumps(payload, indent=2)
    
    try:
        if isinstance(image_input, str):
            img = Image.open(image_input).convert("RGB")
        else:
            img = Image.fromarray(image_input).convert("RGB")
            
        results = trained_model.predict(source=img, conf=conf_threshold, verbose=False)[0]
        annotated_bgr = results.plot()
        annotated_rgb = annotated_bgr[..., ::-1]  # Convert BGR to RGB
        
        counts = {name: 0 for name in CLASS_NAMES}
        for box in results.boxes:
            cls_id = int(box.cls[0])
            if cls_id < len(CLASS_NAMES):
                counts[CLASS_NAMES[cls_id]] += 1
                
        payload = {
            "status": 200,
            "message": "OK",
            "counts": counts
        }
        return annotated_rgb, json.dumps(payload, indent=2)
        
    except Exception as e:
        empty_counts = {name: 0 for name in CLASS_NAMES}
        payload = {
            "status": 415,
            "message": f"Unsupported Media Type / Processing Error: {str(e)}",
            "counts": empty_counts
        }
        return None, json.dumps(payload, indent=2)

# Gradio Interface Setup
demo = gr.Interface(
    fn=build_reply,
    inputs=[
        gr.Image(type="filepath", label="Upload Walkway Image"),
        gr.Slider(minimum=0.1, maximum=0.9, value=0.4, step=0.05, label="Confidence Threshold")
    ],
    outputs=[
        gr.Image(type="numpy", label="Annotated Detection Output"),
        gr.JSON(label="JSON Response Contract")
    ],
    title="People & Luggage Counting Assistant (CA1 Twist 1)",
    description="Upload a pedestrian walkway image to detect directional movement and luggage carrying status."
)

# Launch Gradio App
demo.launch(share=True, inline=True)

### 14. Deployment Testing & Latency Benchmarking
We execute automated error tests to verify system behavior under abnormal inputs (missing file, invalid format) and benchmark warm inference latency to evaluate deployment performance.

In [ ]:
import time

print("=== AUTOMATED ERROR TEST SUITE ===")
test_cases = [
    ("Valid Image", sample_images[0] if sample_images else None),
    ("Missing Input (None)", None),
]

for label, test_input in test_cases:
    if test_input or label == "Missing Input (None)":
        _, json_res = build_reply(test_input)
        res_dict = json.loads(json_res)
        print(f"Test [{label}] -> Status: {res_dict['status']} | Message: {res_dict['message']}")

print("\n=== LATENCY BENCHMARKING ===")
if sample_images:
    test_img = Image.open(sample_images[0])
    
    # Warm-up run
    _ = trained_model.predict(source=test_img, conf=0.40, verbose=False)
    
    # Benchmark 5 runs
    latencies = []
    for _ in range(5):
        t0 = time.perf_counter()
        _ = trained_model.predict(source=test_img, conf=0.40, verbose=False)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)  # Convert to milliseconds
        
    avg_latency = sum(latencies) / len(latencies)
    print(f"Warm Inference Latency (avg of 5 runs): {avg_latency:.2f} ms")